
# **提示词工程**

在本笔记本中，我们将探索提示词工程的基础知识。我们将从安装库先决条件开始。

In [ ]:
!pip install -U langchain
!pip install -U langchain-openai
!pip install -q chromadb
!pip install -q tiktoken
# python 3.14.4

## 请输入自己的API key

In [2]:
import os
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# **提示的结构**

提示可以由多个组件组成：

* 说明
* 外部信息或上下文
* 用户输入或查询
* 输出指示器

并非所有提示都需要所有这些组件，但好的提示通常会使用其中两个或更多组件。让我们更精确地定义它们。

**说明** 告诉模型要做什么，通常它应该如何使用输入和/或外部信息来产生我们想要的输出。

**外部信息或上下文** 是我们手动插入提示、通过矢量数据库（长期记忆）检索或通过其他方式（API 调用、计算等）提取的附加信息。

**用户输入或查询** 通常是系统用户直接输入的查询。

**输出指示器** 是生成文本的 *开头*。对于生成 Python 代码的模型，我们可能会放置“import”（因为大多数 Python 脚本都以库“import”开头），或者聊天机器人可能以“Chatbot:”开头（假设我们将聊天机器人脚本格式化为“User”和“Chatbot”之间交换文本的行）。

通常应按照我们描述的顺序放置这些组件中的每一个。我们从说明开始，提供上下文（如果需要），然后添加用户输入，最后以输出指示器结束。

In [3]:
prompt = """根据以下上下文回答问题。如果无法使用提供的信息回答问题，请回答“我不知道”。

上下文：大型语言模型 (LLM) 是 NLP 中使用的最新模型。
它们比小型模型更出色的性能使它们对于构建支持 NLP 的应用程序的开发人员非常有用。这些模型可以通过 Hugging Face 的 `transformers` 库、通过 OpenAI 使用 `openai` 库以及通过 Cohere 使用 `cohere` 库访问。

问题：哪些库和模型提供商提供 LLM？

答案："""

在此示例中，我们有：

```
说明

上下文

问题（用户输入）

输出指示符（“答案：”）
```

让我们尝试将其发送到 openai 模型

## 我们按如下方式初始化模型：

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="deepseek-v4-flash", base_url="https://api.deepseek.com",
                 api_key="xx")

## 将上面定义的prompt传入做回答

In [4]:
response = llm.invoke(prompt)
print(response.content)

Hugging Face 的 `transformers` 库、OpenAI 的 `openai` 库和 Cohere 的 `cohere` 库。


### 我们通常不会事先知道用户的提示是什么，所以我们实际上想手动添加这个。因此，我们不是直接写提示，而是创建一个带有单个输入变量“query”的“PromptTemplate”。

In [6]:
from langchain_core.prompts import PromptTemplate

template = """根据以下上下文回答问题。如果无法使用提供的信息回答问题，请回答“我不知道”。

上下文：大型语言模型 (LLM) 是 NLP 中使用的最新模型。
它们比小型模型更出色的性能使它们对于构建支持 NLP 的应用程序的开发人员非常有用。这些模型可以通过 Hugging Face 的 `transformers` 库、通过 OpenAI 使用 `openai` 库以及通过 Cohere 使用 `cohere` 库访问。

问题：{query}

答案： """

prompt_template = PromptTemplate(
    input_variables=["query"],
    template=template
)

### 现在我们可以通过“query”参数将用户的“query”插入到提示模板中。

In [7]:
print(
    prompt_template.format(
        query="哪些库和模型提供商提供 LLM？"
    )
)

根据以下上下文回答问题。如果无法使用提供的信息回答问题，请回答“我不知道”。

上下文：大型语言模型 (LLM) 是 NLP 中使用的最新模型。
它们比小型模型更出色的性能使它们对于构建支持 NLP 的应用程序的开发人员非常有用。这些模型可以通过 Hugging Face 的 `transformers` 库、通过 OpenAI 使用 `openai` 库以及通过 Cohere 使用 `cohere` 库访问。

问题：哪些库和模型提供商提供 LLM？

答案： 


In [8]:
print(llm.invoke(
    prompt_template.format(
        query="哪些库和模型提供商提供 LLM？"
    )
))

content='Hugging Face 的 `transformers` 库、OpenAI 的 `openai` 库以及 Cohere 的 `cohere` 库提供 LLM。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 126, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f20662549', 'finish_reason': 'stop', 'logprobs': None} id='run-7fc2a15c-9ca4-46f2-a39e-0678405d83d4-0' usage_metadata={'input_tokens': 126, 'output_tokens': 36, 'total_tokens': 162, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


#### 这只是一个简单的实现，我们可以轻松地用 f 字符串替换它（例如 `f"插入一些自定义文本“{custom_text}”等"`）。但是使用 LangChain 的 `PromptTemplate` 对象，我们能够形式化该过程，添加多个参数，并以面向对象的方式构建提示。

#### 然而，这些并不是使用 LangChains 提示工具的唯一好处。

# **小样本的例子**

LangChain 提供的另一个有用功能是 `FewShotPromptTemplate` 对象。这对于我们使用提示进行所谓的 *少样本学习* 非常理想。

为了提供一些背景信息，LLM 的主要“知识”来源是：

* **参数知识** — 知识是在模型训练期间学习的，并存储在模型权重中。

* **源知识** — 知识在推理时在模型输入中提供，即通过提示。

`FewShotPromptTemplate` 背后的想法是提供少样本训练作为 **源知识**。为此，我们在提示中添加了一些示例，模型可以读取这些示例，然后将其应用于用户的输入。这些都是假装创建反义词的任务的很多例子。

## 小样本训练

### 零样本的问题，

有时我们可能会发现模型似乎没有达到我们的预期。我们可以在以下示例中看到这一点：

In [9]:
prompt = """以下是与 AI 助手的对话。
助手通常很讽刺、很机智，对用户的问题做出有创意、有趣的回答。

用户：什么是优秀的网络工程师？
请回答问题"""

# openai.temperature = 1.0  # increase creativity/randomness of output

print(llm.invoke(prompt))

content='当然，成为一名优秀的网络工程师就像成为技术界的“蜘蛛侠”，只不过你的“蛛丝”是由光纤和以太网电缆组成的。\n\n首先，你需要有扎实的技术基础，掌握网络协议、路由和交换技术。你的任务是确保数据像马戏团的杂技演员一样在网络中优雅地跳跃，而不是像喝醉的熊一样摔倒。\n\n其次，问题解决能力是关键。网络故障不会像坏掉的咖啡机一样自己修好，你得能在最短的时间内找出问题所在，并用一种让人觉得你是“网络巫师”的方式解决问题。\n\n此外，优秀的沟通能力也是必不可少的。你需要能把复杂的技术术语翻译成普通人能理解的语言，让大家都觉得你不仅是技术天才，还很通情达理。\n\n最后，保持学习的热情。网络技术发展迅速，你需要不断更新自己的知识库，仿佛是在追赶一辆永远不会停下的技术火车。\n\n所以，成为一名优秀的网络工程师需要技术、智慧和一点点的幽默感——因为当网络崩溃时，笑是唯一能拯救你的东西。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 281, 'prompt_tokens': 54, 'total_tokens': 335, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f20662549', 'finish_reason': 'stop', 'logprobs': None} id='run-7b0f2bbd-afb6-442c-88be-4990ea2a4da8-0' usage_metadata={'input_tokens': 54, 'output_tokens': 281, 'total_tokens': 335, 'input_to

### 为了帮助改善模型，我们可以给它一些我们想要的答案类型的示例：

In [10]:
prompt = """以下是与 AI 助手对话的摘录。助手通常很讽刺、机智，对用户的问题做出有创意且有趣的回答。以下是一些示例：

用户：我的路由器不工作了

AI：请重新启动路由器好吗？让我给你 cli 命令。

用户：它仍然不工作

AI：请对线路卡进行诊断

用户：我们可以明天做吗？

AI：不，这是一个非常紧急的问题，请稍等片刻

用户：什么是网络工程师

AI： """

print(llm.invoke(prompt))

content='网络工程师是那些在数字世界中搭建和维护“信息高速公路”的人。他们确保我们的数据在互联网上顺畅无阻地传输，就像是数据流动的交通警察和建筑师的结合体。他们的工作是让互联网像魔法一样运作，虽然如果出了问题，他们可能更像是数字侦探，试图找出哪里出了岔子。总之，没有他们，我们就无法享受无缝的在线体验。希望你没有打算立刻去找他们修路由器！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 130, 'total_tokens': 252, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f20662549', 'finish_reason': 'stop', 'logprobs': None} id='run-4e46a404-6693-4711-9024-bfaa1bd29543-0' usage_metadata={'input_tokens': 130, 'output_tokens': 122, 'total_tokens': 252, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 我们现在得到了更好的响应，我们通过*少量学习*通过我们的源知识添加一些示例来实现这一点。

### 现在，要使用 LangChain 的“FewShotPromptTemplate”实现这一点，我们需要执行以下操作：

In [11]:
from langchain import FewShotPromptTemplate

# 创建示例
examples = [
    {
        "query": "我的路由器无法正常工作",
        "answer": "请重启路由器，我会提供CLI命令给您"
    }, {
        "query": "它仍然无法正常工作",
        "answer": "请您对线路卡运行诊断"
    },
    {
        "query": "我们可以明天再处理吗？",
        "answer": "不行，这是一个非常紧急的问题，请稍等一下"
    }
]

# 创建一个示例模板
example_template = """
用户: {query}
AI: {answer}
"""

# 从上述模板创建一个示例提示
example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

# 将之前的提示分为前缀和后缀
# 前缀是我们的说明
prefix = """以下是与AI助手对话的摘录。
这个助手通常带有讽刺和机智的风格，并给出具有创意和幽默的回答。以下是一些示例：
"""
# 后缀是用户输入和输出的指示符
suffix = """
用户: {query}
AI: """

# 现在创建 few-shot 提示模板
few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

### 现在让我们看看当我们输入用户查询时会产生什么……

In [12]:
query = "如何定义一位网络工程师？"

print(few_shot_prompt_template.format(query=query))

以下是与AI助手对话的摘录。
这个助手通常带有讽刺和机智的风格，并给出具有创意和幽默的回答。以下是一些示例：



用户: 我的路由器无法正常工作
AI: 请重启路由器，我会提供CLI命令给您



用户: 它仍然无法正常工作
AI: 请您对线路卡运行诊断



用户: 我们可以明天再处理吗？
AI: 不行，这是一个非常紧急的问题，请稍等一下



用户: 如何定义一位网络工程师？
AI: 


### 为了生成最后结果，我们只需执行以下操作：

In [13]:
print(llm.invoke(
    few_shot_prompt_template.format(query=query)
))

content='一位网络工程师就是那种能够在无数电缆和闪烁的灯光中找到自己的Zen的人，他们能在大家还不明白Wi-Fi和Waffle区别的时候，就已经在调整网络的"味道"了。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 138, 'total_tokens': 193, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d28bcae782', 'finish_reason': 'stop', 'logprobs': None} id='run-862ab28e-c9f1-49ad-828b-e042bf3bf32b-0' usage_metadata={'input_tokens': 138, 'output_tokens': 55, 'total_tokens': 193, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 但是，这确实有些复杂。
为什么要使用`FewShotPromptTemplate`，`examples`字典等进行上述所有操作——当我们可以使用单个f字符串执行相同操作时。

嗯，这种方法更强大，包含一些不错的功能。其中之一就是能够根据查询的长度包含或排除示例。

这实际上非常重要，因为我们的提示和生成输出的最大长度是有限的。这个限制是*最大上下文窗口*，只是我们的提示的长度+我们生成的长度（我们通过`max_tokens`定义）。

因此，我们必须尝试最大化我们给模型的示例数量作为小样本学习示例，同时确保我们不超过最大上下文窗口或过度增加处理时间。

让我们看看示例的动态包含/排除是如何工作的。首先，我们需要更多示例：

In [14]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input": "windy", "output": "calm"},
]

In [15]:
from langchain.prompts.example_selector import LengthBasedExampleSelector
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=30  # 这设置了示例的最大长度
)

### 然后我们使用选择器来初始化一个“dynamic_prompt_template”。

In [16]:
# 现在创建几个镜头提示模板
dynamic_prompt_template = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Give the antonym of every input",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

### 我们可以看到，包含的提示的数量将根据查询的长度而变化......

In [17]:
print(dynamic_prompt_template.format(adjective="big"))

Give the antonym of every input

Input: happy
Output: sad

Input: tall
Output: short

Input: energetic
Output: lethargic

Input: sunny
Output: gloomy

Input: windy
Output: calm

Input: big
Output:


### 问一个更加长的问题

In [19]:
query2 = """big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"""

print(dynamic_prompt_template.format(adjective=query2))

Give the antonym of every input

Input: happy
Output: sad

Input: tall
Output: short

Input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
Output:


### 这样我们就限制了提示中给出的示例数量。如果我们认为这个数量太少，我们可以增加“example_selector”的“max_length”。

# **相似性示例选择器**
SemanticSimilarityExampleSelector 根据哪些示例与输入最相似来选择示例。它通过查找与输入具有最大余弦相似度的嵌入示例来实现这一点。

In [ ]:
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

# 这些都是假装创建反义词的任务的很多例子。
examples_simi = [
    {"input": "开心", "output": "悲伤"},
    {"input": "高大", "output": "短小"},
    {"input": "精力充沛", "output": "昏昏欲睡"},
    {"input": "晴朗", "output": "阴沉"},
    {"input": "有风", "output": "冷静"},
]

In [ ]:
example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 这是可供选择的示例列表。
    examples=examples_simi,
    # 这是用于生成用于测量语义相似度的嵌入的嵌入类。
    embeddings=qwen_embed,
    # 这是用于存储嵌入并进行相似性搜索的 VectorStore 类。
    vectorstore_cls=Chroma,
    # 这是要生成的示例数量。
    k=1
)
similar_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个输入的反义词",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

In [ ]:

print(similar_prompt.format(adjective="忧心忡忡"))

给出每个输入的反义词

Input: 开心
Output: 悲伤

Input: 忧心忡忡
Output:


In [ ]:

print(similar_prompt.format(adjective="多云"))

给出每个输入的反义词

Input: 晴朗
Output: 阴沉

Input: 多云
Output:


In [ ]:

similar_prompt.example_selector.add_example({"input": "热情的", "output": "冷漠的"})
print(similar_prompt.format(adjective="累了"))

给出每个输入的反义词

Input: 精力充沛
Output: 昏昏欲睡

Input: 累了
Output:
